# NFL Upset Prediction: Model Comparison Analysis

This notebook compares XGBoost and LSTM models:
1. Load both trained models
2. Generate predictions on test set
3. Compare metrics (AUC, calibration)
4. Agreement rate analysis
5. Disagreement case studies
6. Feature importance comparison
7. Betting backtest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve

# Project imports
import sys
sys.path.insert(0, '..')

from src.models.lstm_model import UpsetLSTM
from src.evaluation.metrics import EvaluationMetrics
from src.evaluation.comparison import ModelComparison

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Load Models and Data

In [ ]:
# Load XGBoost model
xgb_data = joblib.load('../results/models/xgboost_upset_model.joblib')
xgb_model = xgb_data['model']
xgb_features = xgb_data['feature_columns']
print(f"XGBoost CV AUC: {xgb_data['cv_auc']:.4f}")
print(f"XGBoost features: {len(xgb_features)}")

In [ ]:
# Load LSTM model
lstm_checkpoint = torch.load('../results/models/lstm_upset_model.pt', map_location=device)
lstm_config = lstm_checkpoint['model_config']

lstm_model = UpsetLSTM(**lstm_config)
lstm_model.load_state_dict(lstm_checkpoint['model_state_dict'])
lstm_model.to(device)
lstm_model.eval()

print(f"LSTM CV AUC: {lstm_checkpoint['cv_auc']:.4f}")
print(f"Rolling features: {len(lstm_checkpoint['rolling_features'])}")
print(f"Matchup features: {len(lstm_checkpoint['matchup_features'])}")

In [ ]:
# Load test data (most recent season)
df = pd.read_parquet('../data/processed/features_engineered.parquet')

# Use most recent season as test
test_season = df['season'].max()
test_df = df[df['season'] == test_season].copy()
train_df = df[df['season'] < test_season].copy()

print(f"Train seasons: {train_df['season'].min()} - {train_df['season'].max()}")
print(f"Test season: {test_season}")
print(f"Test games: {len(test_df):,}")

## 2. Generate Predictions

In [ ]:
# Prepare test data
all_features = xgb_features
test_clean = test_df.dropna(subset=all_features + ['is_upset'])
print(f"Test games after dropping missing: {len(test_clean):,}")

X_test = test_clean[xgb_features]
y_test = test_clean['is_upset'].values.astype(int)

In [ ]:
# XGBoost predictions
xgb_preds = xgb_model.predict_proba(X_test)
print(f"XGBoost predictions: min={xgb_preds.min():.3f}, max={xgb_preds.max():.3f}, mean={xgb_preds.mean():.3f}")

In [ ]:
# LSTM predictions
rolling_features = lstm_checkpoint['rolling_features']
matchup_features = lstm_checkpoint['matchup_features']

X_rolling = test_clean[rolling_features].values
X_matchup = test_clean[matchup_features].values

# Create sequence
SEQUENCE_LENGTH = 5
n_rolling = len(rolling_features)
X_seq = X_rolling.reshape(len(X_rolling), 1, n_rolling)
X_seq = np.repeat(X_seq, SEQUENCE_LENGTH, axis=1)

# Scale
scaler_seq = StandardScaler()
scaler_seq.mean_ = lstm_checkpoint['scaler_seq_mean']
scaler_seq.scale_ = lstm_checkpoint['scaler_seq_scale']

scaler_matchup = StandardScaler()
scaler_matchup.mean_ = lstm_checkpoint['scaler_matchup_mean']
scaler_matchup.scale_ = lstm_checkpoint['scaler_matchup_scale']

X_seq_scaled = scaler_seq.transform(X_seq.reshape(-1, n_rolling)).reshape(X_seq.shape)
X_matchup_scaled = scaler_matchup.transform(X_matchup)

# Predict
with torch.no_grad():
    X_seq_t = torch.FloatTensor(X_seq_scaled).to(device)
    X_matchup_t = torch.FloatTensor(X_matchup_scaled).to(device)
    lstm_preds = lstm_model(X_seq_t, X_matchup_t).cpu().numpy().flatten()

print(f"LSTM predictions: min={lstm_preds.min():.3f}, max={lstm_preds.max():.3f}, mean={lstm_preds.mean():.3f}")

## 3. Compare Metrics

In [ ]:
# Evaluate both models
xgb_metrics = EvaluationMetrics(y_test, xgb_preds)
lstm_metrics = EvaluationMetrics(y_test, lstm_preds)

comparison_df = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Brier Score', 'Accuracy', 'Log Loss'],
    'XGBoost': [
        xgb_metrics.auc_roc(),
        xgb_metrics.brier_score(),
        xgb_metrics.accuracy(),
        xgb_metrics.log_loss(),
    ],
    'LSTM': [
        lstm_metrics.auc_roc(),
        lstm_metrics.brier_score(),
        lstm_metrics.accuracy(),
        lstm_metrics.log_loss(),
    ]
})
comparison_df['Difference'] = comparison_df['XGBoost'] - comparison_df['LSTM']
comparison_df['Better'] = comparison_df.apply(
    lambda row: 'XGBoost' if (row['Difference'] > 0 and row['Metric'] == 'AUC-ROC') or 
                            (row['Difference'] < 0 and row['Metric'] in ['Brier Score', 'Log Loss']) 
                else 'LSTM' if (row['Difference'] < 0 and row['Metric'] == 'AUC-ROC') or
                               (row['Difference'] > 0 and row['Metric'] in ['Brier Score', 'Log Loss'])
                else 'Tie', axis=1
)

print("Model Comparison on Test Set:")
print(comparison_df.to_string(index=False))

In [ ]:
# Calibration curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost calibration
prob_true_xgb, prob_pred_xgb = calibration_curve(y_test, xgb_preds, n_bins=10)
axes[0].plot(prob_pred_xgb, prob_true_xgb, 'o-', label='XGBoost')
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect')
axes[0].set_xlabel('Predicted Probability')
axes[0].set_ylabel('True Probability')
axes[0].set_title('XGBoost Calibration Curve')
axes[0].legend()

# LSTM calibration
prob_true_lstm, prob_pred_lstm = calibration_curve(y_test, lstm_preds, n_bins=10)
axes[1].plot(prob_pred_lstm, prob_true_lstm, 'o-', label='LSTM', color='orange')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect')
axes[1].set_xlabel('Predicted Probability')
axes[1].set_ylabel('True Probability')
axes[1].set_title('LSTM Calibration Curve')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/figures/calibration_comparison.png', dpi=150)
plt.show()

## 4. Agreement Analysis

In [ ]:
# Prediction agreement
threshold = 0.5
xgb_class = (xgb_preds >= threshold).astype(int)
lstm_class = (lstm_preds >= threshold).astype(int)

agreement = (xgb_class == lstm_class).mean()
print(f"Classification agreement rate: {agreement:.1%}")

# Prediction correlation
pred_corr = np.corrcoef(xgb_preds, lstm_preds)[0, 1]
print(f"Prediction correlation: {pred_corr:.3f}")

In [ ]:
# Scatter plot of predictions
fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(xgb_preds, lstm_preds, c=y_test, cmap='RdYlGn', alpha=0.6)
ax.plot([0, 1], [0, 1], 'k--', label='y=x')
ax.set_xlabel('XGBoost Predicted Probability')
ax.set_ylabel('LSTM Predicted Probability')
ax.set_title(f'Prediction Comparison (r={pred_corr:.3f})')
ax.legend()
plt.colorbar(scatter, label='Actual Upset (1=Yes)')
plt.tight_layout()
plt.savefig('../results/figures/prediction_scatter.png', dpi=150)
plt.show()

In [ ]:
# Agreement matrix
confusion = pd.crosstab(
    pd.Series(xgb_class, name='XGBoost Predicts Upset'),
    pd.Series(lstm_class, name='LSTM Predicts Upset'),
    margins=True
)
print("\nPrediction Agreement Matrix:")
print(confusion)

## 5. Disagreement Case Studies

In [ ]:
# Find games where models disagree
test_clean = test_clean.copy()
test_clean['xgb_pred'] = xgb_preds
test_clean['lstm_pred'] = lstm_preds
test_clean['xgb_class'] = xgb_class
test_clean['lstm_class'] = lstm_class
test_clean['disagree'] = xgb_class != lstm_class

disagree_games = test_clean[test_clean['disagree']].copy()
print(f"Games where models disagree: {len(disagree_games)} ({len(disagree_games)/len(test_clean):.1%})")

In [ ]:
# XGBoost predicts upset, LSTM doesn't
xgb_only_upset = disagree_games[disagree_games['xgb_class'] == 1]
print(f"\nXGBoost predicts upset, LSTM doesn't: {len(xgb_only_upset)}")
print(f"Actual upset rate: {xgb_only_upset['is_upset'].mean():.1%}")

# LSTM predicts upset, XGBoost doesn't
lstm_only_upset = disagree_games[disagree_games['lstm_class'] == 1]
print(f"\nLSTM predicts upset, XGBoost doesn't: {len(lstm_only_upset)}")
print(f"Actual upset rate: {lstm_only_upset['is_upset'].mean():.1%}")

In [ ]:
# Show interesting disagreement cases
if len(disagree_games) > 0:
    display_cols = ['season', 'week', 'home_team', 'away_team', 'home_score', 'away_score', 
                    'spread_magnitude', 'is_upset', 'xgb_pred', 'lstm_pred']
    display_cols = [c for c in display_cols if c in disagree_games.columns]
    
    print("\nSample disagreement cases:")
    print(disagree_games[display_cols].head(10).to_string())

## 6. Feature Importance Comparison

In [ ]:
# Compare XGBoost feature importance (SHAP) with built-in importance
shap_importance = xgb_data.get('shap_importance', {})
xgb_importance = xgb_data.get('xgb_importance', {})

if shap_importance:
    importance_compare = pd.DataFrame({
        'feature': list(shap_importance.keys()),
        'shap': list(shap_importance.values()),
    })
    if xgb_importance:
        importance_compare['xgb_native'] = importance_compare['feature'].map(xgb_importance)
    
    importance_compare = importance_compare.sort_values('shap', ascending=False)
    
    # Plot top features
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = 15
    top_features = importance_compare.head(top_n)
    
    y_pos = range(len(top_features))
    ax.barh(y_pos, top_features['shap'], color='steelblue', alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_features['feature'])
    ax.set_xlabel('SHAP Feature Importance')
    ax.set_title('Top 15 XGBoost Features (SHAP Importance)')
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig('../results/figures/xgboost_top_features.png', dpi=150)
    plt.show()
else:
    print("SHAP importance not available in saved model")

## 7. Betting Backtest

In [ ]:
# Simple betting strategy: bet on upset when model predicts > threshold
def betting_backtest(predictions, actuals, spread_magnitudes, threshold=0.5, unit_size=100):
    """Backtest betting on model predictions."""
    results = {
        'bets': 0,
        'wins': 0,
        'losses': 0,
        'profit': 0,
        'roi': 0,
    }
    
    for pred, actual, spread in zip(predictions, actuals, spread_magnitudes):
        if pred >= threshold:  # Bet on upset
            results['bets'] += 1
            
            # Simplified: assume ~+200 average odds for underdog ML
            # Higher spread = higher potential payout
            odds = 100 + (spread * 15)  # Rough approximation
            
            if actual == 1:  # Upset occurred
                results['wins'] += 1
                results['profit'] += unit_size * (odds / 100)
            else:
                results['losses'] += 1
                results['profit'] -= unit_size
    
    if results['bets'] > 0:
        results['win_rate'] = results['wins'] / results['bets']
        results['roi'] = results['profit'] / (results['bets'] * unit_size)
    
    return results

In [ ]:
# Backtest both models
spread_mags = test_clean['spread_magnitude'].values if 'spread_magnitude' in test_clean.columns else np.full(len(y_test), 7.0)

thresholds = [0.3, 0.4, 0.5, 0.6]
backtest_results = []

for thresh in thresholds:
    xgb_bt = betting_backtest(xgb_preds, y_test, spread_mags, threshold=thresh)
    lstm_bt = betting_backtest(lstm_preds, y_test, spread_mags, threshold=thresh)
    
    backtest_results.append({
        'Threshold': thresh,
        'XGBoost Bets': xgb_bt['bets'],
        'XGBoost Win%': xgb_bt.get('win_rate', 0),
        'XGBoost ROI': xgb_bt['roi'],
        'LSTM Bets': lstm_bt['bets'],
        'LSTM Win%': lstm_bt.get('win_rate', 0),
        'LSTM ROI': lstm_bt['roi'],
    })

bt_df = pd.DataFrame(backtest_results)
print("Betting Backtest Results:")
print(bt_df.to_string(index=False))

In [ ]:
# Visualize backtest results
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(thresholds))
width = 0.35

ax.bar([i - width/2 for i in x], bt_df['XGBoost ROI'], width, label='XGBoost', color='steelblue', alpha=0.7)
ax.bar([i + width/2 for i in x], bt_df['LSTM ROI'], width, label='LSTM', color='coral', alpha=0.7)

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.set_xlabel('Prediction Threshold')
ax.set_ylabel('ROI')
ax.set_title('Betting Backtest: ROI by Model and Threshold')
ax.set_xticks(x)
ax.set_xticklabels(thresholds)
ax.legend()

plt.tight_layout()
plt.savefig('../results/figures/betting_backtest.png', dpi=150)
plt.show()

## 8. Summary Report

In [ ]:
print("=" * 60)
print("NFL UPSET PREDICTION: MODEL COMPARISON SUMMARY")
print("=" * 60)

print(f"\nTEST SET: Season {test_season} ({len(test_clean):,} games)")
print(f"Upset rate: {y_test.mean():.1%}")

print(f"\nMODEL PERFORMANCE:")
print(comparison_df.to_string(index=False))

print(f"\nAGREEMENT:")
print(f"  Classification agreement: {agreement:.1%}")
print(f"  Prediction correlation: {pred_corr:.3f}")

print(f"\nKEY FINDINGS:")
winner_auc = 'XGBoost' if xgb_metrics.auc_roc() > lstm_metrics.auc_roc() else 'LSTM'
winner_brier = 'XGBoost' if xgb_metrics.brier_score() < lstm_metrics.brier_score() else 'LSTM'
print(f"  - Better AUC-ROC: {winner_auc}")
print(f"  - Better Brier Score: {winner_brier}")

if len(disagree_games) > 0:
    print(f"  - Disagreement games: {len(disagree_games)} ({len(disagree_games)/len(test_clean):.1%})")
    
print(f"\nFigures saved to: results/figures/")